# Linear regression

Linear regression fits a straight line $y = wx + b$ to data by minimizing
squared error. It's the "hello world" of supervised learning. We'll use
[`linfa-linear`](https://docs.rs/linfa-linear), the `linfa` counterpart to
scikit-learn's `LinearRegression`.

This uses the `ndarray` arrays from [Chapter 1](../01-foundations/ndarray-basics.ipynb).

In [ ]:
:dep ndarray = { version = "0.15" }
:dep linfa = { version = "0.7" }
:dep linfa-linear = { version = "0.7" }
use ndarray::array;

// Feature matrix `x` (one column) and target vector `y`. The true relationship
// is roughly y = 2x, with a little noise.
let x = array![[1.0_f64], [2.0], [3.0], [4.0], [5.0]];
let y = array![2.1_f64, 3.9, 6.2, 7.8, 10.1];
println!("{} samples, {} feature(s)", x.nrows(), x.ncols());

## Fit the model

`linfa` wraps features + targets in a `Dataset`. We fit inside a `{ }` block
and return just the learned slope and intercept (plain `f64` values), because
the fitted-model type can't be persisted across cells (see the
[crate reference](../appendix/crate-reference.md)).

In [ ]:
use linfa::prelude::*;
use linfa::Dataset;
use linfa_linear::LinearRegression;

// Explicit `: (f64, f64)` so evcxr can persist these across cells.
let (slope, intercept): (f64, f64) = {
    let dataset = Dataset::new(x.clone(), y.clone());
    let model = LinearRegression::default().fit(&dataset).expect("fit failed");
    (model.params()[0], model.intercept())
};
println!("learned slope     = {:.3}", slope);
println!("learned intercept = {:.3}", intercept);
println!("prediction at x=6 : {:.3}", slope * 6.0 + intercept);

The slope is close to 2 and the intercept close to 0 — the model recovered the
underlying $y \approx 2x$ relationship.

## Visualize the fit

Plotting the data points against the fitted line, using
[`plotters`](https://docs.rs/plotters):

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

evcxr_figure((480, 320), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("Linear regression fit", ("sans-serif", 18))
        .margin(10)
        .x_label_area_size(30)
        .y_label_area_size(40)
        .build_cartesian_2d(0f64..6f64, 0f64..12f64)?;
    chart.configure_mesh().draw()?;
    // Data points (red).
    chart.draw_series(
        (0..x.nrows()).map(|i| Circle::new((x[[i, 0]], y[i]), 4, RED.filled())),
    )?;
    // Fitted line (blue).
    chart.draw_series(LineSeries::new(
        (0..=60).map(|t| { let xv = t as f64 / 10.0; (xv, slope * xv + intercept) }),
        &BLUE,
    ))?;
    Ok(())
})

Next: [logistic regression](logistic-regression.ipynb) — the same idea, but for
predicting **categories** instead of continuous values.